# 01. Data Integrity and Return Math

Snapshots: `2026-09-23, 2026-09-30`

> **Status (2026-10-13): in progress.** Task 5 steps 1 and 2 of the
> [week 1 brief](../briefs/01-data-integrity.md) are done. Still open:
>
> - Concept and derivation: the second-order expansion (point 2), what an adjusted
>   close is and why it restates history (point 4), and the snapshot-replay paragraph
>   (point 5). The compounding identity is written with `+` where it should be a product.
> - Exercise 2: Sharpe two ways for all eight tickers, with and without the DTB3
>   risk-free rate (brief step 3).
> - Exercise 1: the restatement diff between the 2026-09-23 and 2026-09-30 snapshots,
>   matched to that week's ex-dividend dates (brief steps 4 and 5).
> - The `as_of` comparison run of `configs/week1.yaml` (brief step 6).
> - Sanity check: log-return and annualised-mean statistics alongside the simple ones.
> - The two exercise tables in "What the numbers showed"; the Reading section.
> - Cell 9 fetches the unadjusted SPY close live; hardcode the quoted price so the
>   notebook runs offline.


## The question this lesson answers

How do you know how well a stock performed? 
1. Use snapshots that represent only what you would have known at decision time
2. Calculate returns correctly -- it's just math.

## Concept and derivation

### Return identities

Let $P_t$ be the total-return adjusted close on day $t$.

- simple:  $r_t = \frac{P_t}{P_{t-1}} - 1$
- log:     $l_t$ = $\ln(P_t / P_{t-1}) = \ln(P_t) - \ln(P_{t-1})$

Log returns **add**; they telescope over any horizon:

$\sum_1^T(l_t) = \ln P_T - \ln P_0 = \ln(P_T / P_0)$

Simple returns **compound**: 
$\prod_1^T (1 + r_t) = (1 + \frac{P_T}{P_{T-1}} - 1) + \cdots + (1+ \frac{P_1}{P_0} - 1) = P_T / P_0$. 

The first terms of the Taylor series expansion of the log returns give that

$\ln(1 + r) \approx r - r^2/2 \rightarrow mean(log) \approx mean(simple) - \sigma^2/2$

That asymmetry between adding and compounding is important, and the $\sigma^2/2$ term shows how different they are.
Take a price path of 100 -> 150 -> 100: the simple returns are +50% and -33.3%,
whose mean is positive, so averaging them says I made money on a round trip. The log
returns are +0.4055 and -0.4055, which sum to exactly zero and say I am flat. Logs are
the honest summary over time; simple returns are what compounds into a horizon figure.

Both are one value per `(date, ticker)` — the same shape as the price frame, with a NaN
first row because day 0 has no predecessor. In pandas that falls out of `prices.shift(1)`,
which moves values down one row while the index stays put, so each row holds yesterday's
price against today's date and the division aligns itself. Since `diff()` is defined as
`x - x.shift(1)`, `np.log(prices).diff()` is the entire log-return function.

### Variance, volatility, and the square root of time

Variance is the mean squared deviation of returns. Its units are return-squared, so it is
not directly interpretable. Volatility is its square root — the standard deviation, in the
same units as the returns themselves.

If daily returns are independent, variance adds over time: the variance of a 252-day sum
is 252x the one-day variance. Volatility is the square root of variance, so

```
sigma_annual = sqrt(252 * sigma_daily^2) = sigma_daily * sqrt(252)
```

This makes sense, because a ticker accumulates randomness over time — but it accumulates
*with cancellation*. If today's price is yesterday's plus a random deviation, the
deviations pile up, yet each one carries a random sign, so some undo others. The algebra
that encodes this is `Var(X + Y) = Var(X) + Var(Y) + 2·Cov(X, Y)`: when the days are
independent the covariance term is zero, the variances simply add, and the square root
that converts back to return units is where the `sqrt(T)` comes from. Note that volatility
does not *compound* — compounding is what would give growth proportional to `T`, and that
is the answer you get only if every day moves the same direction. Simulating 252
independent 1% days bears this out: about 95% of the daily motion cancels, leaving a
typical annual move near 15.9% rather than 252%.

## The experiment

What data, what config, what was varied, what you expected before running it.

Load the wide prices and find:
- How many trading days per ticker? 2953 for all of them. Do all eight start on 2015-01-02? Yes.

In [1]:
from portfolio_lab.data.store import ParquetDuckStore

store = ParquetDuckStore()
prices = store.load_prices()

days = prices.count()
print(days)

start_dates = prices.apply(lambda col: col.first_valid_index())
print(start_dates)

AAPL    2953
EFA     2953
GLD     2953
IEF     2953
JNJ     2953
MSFT    2953
SPY     2953
XOM     2953
dtype: int64
AAPL   2015-01-02
EFA    2015-01-02
GLD    2015-01-02
IEF    2015-01-02
JNJ    2015-01-02
MSFT   2015-01-02
SPY    2015-01-02
XOM    2015-01-02
dtype: datetime64[us]


- Is SPY adjusted close on 2015-01-02 equal to the price you can find quoted for that day? (It will not be. Explain the gap.)
the store loads prices adjusted for dividends paid since. The price on 2015-01-02 is not what anyone paid then, but
the amount that you would have needed to invest, with dividends reinvested, to own one share today.

In [2]:
from portfolio_lab.data.store import ParquetDuckStore
import yfinance as yf
store = ParquetDuckStore()
spy = yf.download("SPY", start="2015-01-1", auto_adjust=False)["Close"]
prices = store.load_prices()
adjusted = prices.loc["2015-01-02", "SPY"]
unadjusted = spy.loc["2015-01-02", "SPY"]
print(f"adjusted close on 2015-01-2 (adjusted): {adjusted}")
print(f"unadjusted close on 2015-01-02 (not adjusted): {unadjusted}")
print(f"ratio: {unadjusted / adjusted}")


[*********************100%***********************]  1 of 1 completed

adjusted close on 2015-01-2 (adjusted): 169.2675323486328
unadjusted close on 2015-01-02 (not adjusted): 205.42999267578125
ratio: 1.2136408549553745


- Compute simple_returns and log_returns; report mean and std of each for SPY and AAPL, daily and annualised.

In [3]:
from portfolio_lab.data.store import ParquetDuckStore
from portfolio_lab.data.returns import simple_returns, log_returns, annualise_vol
store = ParquetDuckStore()
prices = store.load_prices()
simple = simple_returns(prices)
log = log_returns(prices)
means = simple[["AAPL", "SPY"]].mean()
stds = simple[["AAPL", "SPY"]].std()
annual = annualise_vol(simple[["AAPL", "SPY"]])
print(means, stds, annual)

AAPL    0.001052
SPY     0.000571
dtype: float64 AAPL    0.018087
SPY     0.011045
dtype: float64 AAPL    0.287127
SPY     0.175340
dtype: float64


## What the numbers showed
- In our snapshot, all tickers were traded the same number of days, starting on the same day. Our job is easier.
- SPY's adjusted close on 2015-01-02 is 82% of the price it actually traded on that day. This means that with dividends reinvested, one share owned in 2015 would be 1.2 shares today.
- 

## What went wrong and what I'd do differently

**I wrote a loop where pandas wanted a vectorised expression.** My first `log_returns`
allocated `np.zeros(shape=(prices.shape[0], 1))` and accumulated over rows. Three things
were wrong, in increasing order of how long they would have taken to find:

- `prices[i]` does not select row `i`. `[]` on a DataFrame selects *columns* by label, so
  against ticker columns this is a `KeyError` rather than a wrong number.
- `returns += ...` accumulates everything into one bucket instead of storing a value per day.
- the hardcoded second dimension of `1` would have broken silently on a third ticker.

The fix was to delete the loop, not repair it. `np.log(prices).diff()` is the whole
function: it keeps the index and columns aligned, produces the NaN first row for free, and
works for any number of tickers. I should reach for a frame-level operation first and only
drop to a loop when I can say why the vectorised form cannot express it.

## Pitfalls to remember

- **`ddof`.** `pd.Series.std()` divides by n-1; `np.std()` divides by n. On the 200-point
  test fixture that is 0.01002509 vs 0.01000000, annualising to 15.914% vs 15.875%.
  `pytest.approx` defaults to a relative tolerance of 1e-6 and rejects the numpy one — and
  the gap is small enough to look like rounding and send you hunting in the wrong place.
- **NaN.** `pd.Series.std()` skips NaN; `np.std()` propagates it. Every return frame has a
  NaN first row, so numpy hands back `nan` for the entire series. Stay in pandas.
- **Shift direction.** `shift(1)` reaches backward in time and is what you want.
  `shift(-1)` pulls tomorrow's value onto today's row and is the canonical lookahead bug.
  Say the sign out loud in words before trusting it.
- **Volatility is not volume.** Volume is shares traded. Volatility is the dispersion of
  returns.

## Reading

What you actually read, with the one idea each source contributed.